## Paparella Results

#### TLDR: clone this repo and use runs directory as tensorboard logdir to see Paparellas results.

Paparella et. al. published results of their models only for  
certain time-points and each model was evaluated in different  
time point selected based on ndcg@10 value. However, vanilla models  
hit their highest accuracy early on in training (while diversity and  
novelty are still growing), while SMORL models hit their highest  
accuracy much later in training process (where diversity and novelty  
reach their maximums). This cause that diversity, novelty and likely  
also repetitiveness of vanilla models may be underestimated.

#### Published and "Published" data 
As mentioned above, Paprella published metrics only for certain time points of training  
process in the paper (i.e. in tables and graphs). However, authors also published code  
with various data files including text files that contains almost time series of metrics  
from training of model. I wrote almost as it is plain text that needs to be further processed  
to get time series. 

This notebook iterates through .txt output files, processes them and provide time series of  
accuracy, diversity, novelty and repetitiveness metrices as well as loss in tensorboard format.

In [13]:
import os
import re
import pandas as pd
from networkx.algorithms.bipartite.basic import color

models = ['sasrec', 'caser', 'gru']
combs = [[1,1,1],[1,1,0],[1,0,1],[0,1,1],[0,1,0],[0,0,1]]
model_stop = ["main", "target"]
datasets = ["rc15_results", "retail_rocket_results"]
# MODIFY PATH TO div4rec dir of Paparellas
div4rec_path = "/home/marek/Kinit/MORSs/SMORL/div4rec"
results_dir = "/home/marek/Kinit/my_smorl/Plots/Paparella_both"
os.makedirs(results_dir, exist_ok=True)


In [14]:
patterns = {
    # Matches: cumulative reward @ 5: 8585.000000
    'cumulative_reward': re.compile(r'.*cumulative reward @ (\d+): ([\d.]+)$'),

    # Matches: clicks hr ndcg @ 10 : 0.398485, 0.241570
    'clicks_hr_ndcg': re.compile(r'.*clicks hr ndcg @ (\d+) ?: ([\d.]+), ([\d.]+)$'),

    # Matches: purchase hr and ndcg @10 : 0.535500, 0.337177
    'purchase_hr_ndcg': re.compile(r'.*purchase hr and ndcg @(\d+) ?: ([\d.]+), ([\d.]+)$'),

    # Matches: total diversity reward: 48740.039062
    'total_diversity': re.compile(r'.*total diversity reward: ([\d.]+)$'),

    # Matches: total novelty reward: 19004.000000
    'total_novelty': re.compile(r'.*total novelty reward: ([\d.]+)$'),

    # Matches: coverage of top 5 predictions: 0.379410
    'coverage': re.compile(r'.*coverage of top (\d+) predictions: ([\d.]+)$'),

    # Matches: coverage on novel items of top 10 predictions: 0.385095
    'novel_coverage': re.compile(r'.*coverage on novel items of top (\d+) predictions: ([\d.]+)$'),

    # Matches: average number of repetitions in top 20: 53.379500
    'avg_repetitions': re.compile(r'.*average number of repetitions in top (\d+): ([\d.]+)$'),
    
    'steps': re.compile(r'.*Step: (\d+)\.+\s+Loss: ([\d.]+)$'),
    'Slosses': re.compile(r'.*Supervised loss is ([\d.]+), SMORL loss is ([\d.]+)'),
    'Mlosses': re.compile(r'.*Supervised loss is ([\d.]+), MORL loss is ([\d.]+)')
}

In [77]:
def process_results(data_path, results_path, variant):
    loss = {
        'steps': [], 'loss': [], 'plain': [], 'smorl': []
    }
    metrics = {
        'steps': [],
        'hr_test_5': [], 'hr_test_10': [], 'hr_test_20': [],
        'ndcg_test_5': [], 'ndcg_test_10': [], 'ndcg_test_20': [],
        'cov_test_1': [], 'cov_test_5': [], 'cov_test_10': [], 'cov_test_20': [],
        'nov_test_1': [], 'nov_test_5': [], 'nov_test_10': [], 'nov_test_20': [],
        'rep_test_5': [], 'rep_test_10': [], 'rep_test_20': [],        
        'hr_val_5': [], 'hr_val_10': [], 'hr_val_20': [], 
        'ndcg_val_5': [], 'ndcg_val_10': [], 'ndcg_val_20': [],
        'cov_val_1': [], 'cov_val_5': [], 'cov_val_10': [], 'cov_val_20': [],        
        'nov_val_1': [], 'nov_val_5': [], 'nov_val_10': [], 'nov_val_20': [],
        'rep_val_5': [], 'rep_val_10': [], 'rep_val_20': [] 
    }
    with open(data_path, "r") as f:
        lines = f.readlines()
        my_model = "any"  
        for line in lines:
            #match = re.search(r"Step: (\d+)\.+\s+Loss: ([\d.]+)", line)
            if m := patterns['Slosses'].search(line):
                plain, smorl = float(m.group(1)), float(m.group(2))
                loss['plain'].append(plain)
                loss['smorl'].append(smorl)
            if m := patterns['Mlosses'].search(line):
                plain, smorl = float(m.group(1)), float(m.group(2))
                loss['plain'].append(plain)
                loss['smorl'].append(smorl)
            if m := patterns['steps'].match(line):
                step, _loss = int(m.group(1)), float(m.group(2))
                loss['steps'].append(step)
                loss['loss'].append(_loss)
            if "Model is" in line:
                my_model = "any"
            if "Evaluating Target Model" in line:
                my_model = "target"
            if "Evaluating Main Model" in line:
                my_model = "main"
            if my_model == variant:
                continue   
            if "TEST" in line:
                if m := patterns['clicks_hr_ndcg'].match(line):
                    k, hr, ndcg = int(m.group(1)), float(m.group(2)), float(m.group(3))
                    metrics[f'hr_test_{k}'].append(hr)
                    metrics[f'ndcg_test_{k}'].append(ndcg)
                elif m := patterns['coverage'].match(line):
                    k, val = int(m.group(1)), float(m.group(2))
                    metrics[f'cov_test_{k}'].append(val)
                elif m := patterns['novel_coverage'].match(line):
                    k, val = int(m.group(1)), float(m.group(2))
                    metrics[f'nov_test_{k}'].append(val)
                elif m := patterns['avg_repetitions'].match(line):
                    k, val = int(m.group(1)), float(m.group(2))
                    metrics[f'rep_test_{k}'].append(val)
            else:
                if m := patterns['clicks_hr_ndcg'].match(line):
                    k, hr, ndcg = int(m.group(1)), float(m.group(2)), float(m.group(3))
                    metrics[f'hr_val_{k}'].append(hr)
                    metrics[f'ndcg_val_{k}'].append(ndcg)
                elif m := patterns['coverage'].match(line):
                    k, val = int(m.group(1)), float(m.group(2))
                    metrics[f'cov_val_{k}'].append(val)
                elif m := patterns['novel_coverage'].match(line):
                    k, val = int(m.group(1)), float(m.group(2))
                    metrics[f'nov_val_{k}'].append(val)
                elif m := patterns['avg_repetitions'].match(line):
                    k, val = int(m.group(1)), float(m.group(2))
                    metrics[f'rep_val_{k}'].append(val)
        metrics['steps'] = [5000 * (i + 1) for i in range(len(metrics['hr_test_5']))]
    
    return loss, metrics

models = ['sasrec', 'caser', 'gru']

for model in models:
    for comb in combs:
        for variant in model_stop:
            for dats in datasets:
                if "rc15" in dats:
                    file_path = f'{div4rec_path}/{dats}/{model}_smorl/{model}_smorl1_acc{comb[0]}.0_div{comb[1]}.0_nov{comb[2]}.0_weighted_q_vals.txt'
                else:
                    file_path = f'{div4rec_path}/{dats}/{model}smorl/{model}_smorl1_acc{comb[0]}.0_div{comb[1]}.0_nov{comb[2]}.0_weighted_q_vals.txt'
                results_path = f'{results_dir}/{dats}/{model}'
                os.makedirs(results_path, exist_ok=True)
                results = f"{results_path}/rl_{comb[0]}{comb[1]}{comb[2]}_{variant}"
                losses, metrics = process_results(file_path, results, variant)
                df_loss = pd.DataFrame(losses)
                df_metrics = pd.DataFrame(metrics)
                df_loss.to_pickle(f'{results}_loss')
                df_loss.to_csv(f'{results}_loss.csv')
                df_metrics.to_pickle(f'{results}_metrics')
                df_metrics.to_csv(f'{results}_metrics.csv')
                
models.append('nextitnet')
for model in models:
    for dats in datasets:
        file_path = f'/home/marek/Kinit/MORSs/SMORL/div4rec/{dats}/{model}/{model}.txt'
        results = f"{results_dir}/{dats}/{model}"
        os.makedirs(results, exist_ok=True)
        losses, metrics = process_results(file_path, results, "This is not rl - not distinguished")
        losses = {k: v for k, v in losses.items() if k not in {'plain', 'smorl'}}
        df_loss = pd.DataFrame(losses)
        df_metrics = pd.DataFrame(metrics)
        df_loss.to_pickle(f'{results}/base_loss')
        df_loss.to_csv(f'{results}/base_loss.csv')
        df_metrics.to_pickle(f'{results}/base_metrics')
        df_metrics.to_csv(f'{results}/base_metrics.csv')


model = "nextitnet"
for variant in model_stop:
    for dats in datasets:
        if "rc15" in dats:
            file_path = f'{div4rec_path}/{dats}/{model}_smorl/{model}_smorl1_acc1.0_div1.0_nov1.0.txt'
        else:
            file_path = f'{div4rec_path}/{dats}/{model}smorl/{model}_smorl1_acc1.0_div1.0_nov1.0.txt'
        results = f"{results_dir}/{dats}/{model}/rl_111_{variant}"
        losses, metrics = process_results(file_path, results, variant)
        df_loss = pd.DataFrame(losses)
        df_metrics = pd.DataFrame(metrics)
        df_loss.to_pickle(f'{results}_base_loss')
        df_loss.to_csv(f'{results}_base_loss.csv')  
        df_metrics.to_pickle(f'{results}_base_metrics')
        df_metrics.to_csv(f'{results}_base_metrics.csv')

In [248]:
points = {
    'rc15_results': {
        'gru': { 
            'base': 10000,
            '111': 10000,
            '001': 25000,
            '010': 25000,
            '011': 30000,
            '110': 25000,
            '101': 25000
        },
        'caser': {
            'base': 10000,
            '111': 75000,
            '001': 90000,
            '010': 40000,
            '011': 30000,
            '110': 110000,
            '101': 130000    
        },
        'sasrec': {
            'base': 25000,
            '111': 105000,
            '001': 115000,
            '010': 130000,
            '011': 115000,
            '110': 165000,
            '101': 115000  
        }
    }
}

In [ ]:
import pandas as pd
from matplotlib import pyplot as plt

def plot_metrics(basepath, dataset, model, replica):
    filepath = f"{basepath}/{dataset}/{model}"
    dataline = "metrics"
    df0 = pd.read_pickle(f"{filepath}/base_{dataline}")
    df1 = pd.read_pickle(f"{filepath}/rl_001_{replica}_{dataline}")
    df2 = pd.read_pickle(f"{filepath}/rl_010_{replica}_{dataline}")
    df3 = pd.read_pickle(f"{filepath}/rl_011_{replica}_{dataline}")
    df4 = pd.read_pickle(f"{filepath}/rl_110_{replica}_{dataline}")
    df5 = pd.read_pickle(f"{filepath}/rl_101_{replica}_{dataline}")
    df6 = pd.read_pickle(f"{filepath}/rl_111_{replica}_{dataline}")
   
    #~~~~~~ Plot cov + nov values ~~~~~~
    PLOT = "cov_val_10"
    fig, axs = plt.subplots(3, 2, figsize=(14, 14))
    fig.suptitle(f'{dataset}-{model}-{replica}-{dataline}', fontsize=12, y=0.94)
    ax1 = axs[0, 0]
    ax1.tick_params(axis='y', labelsize=8)
    ax1.set_title("COV 10 / NOV 10", fontsize=10, pad=10)
    ax1.set_ylabel(PLOT, color='gray')
    ax1.set_ylim(0.1, 0.75)
    ax1.yaxis.grid(True, color='lightgray', linewidth=0.5)
    ax1.axvline(x=points[dataset][model]['111'], color='gray', linewidth=1)
    ax1.axvline(x=points[dataset][model]['base'], color='gray', linestyle="--", linewidth=1)
    
    ax1.plot(df0['steps'], df0[PLOT].rolling(window).mean(), color='black', linestyle=":", linewidth=2)
    for frame in [df1, df2, df3, df4, df5]:
        ax1.plot(frame['steps'], frame[PLOT].rolling(window).mean(), color='lightgray', linewidth=1)
    ax1.plot(df6['steps'], df6[PLOT].rolling(window).mean(), color='black', linewidth=1)

    PLOT = "nov_val_10"
    ax2 = ax1.twinx()
    ax2.set_ylabel(PLOT, color='green')
    ax2.set_ylim(0.1, 0.75)
    ax2.tick_params(axis='y', labelsize=8)

    ax2.plot(df0['steps'], df0[PLOT].rolling(window).mean(), color='green', linestyle='--', linewidth=1)
    for frame in [df1, df2, df3, df4, df5]:
        ax2.plot(frame['steps'], frame[PLOT].rolling(window).mean(), color='lightgreen', linewidth=1)
    ax2.plot(df6['steps'], df6[PLOT].rolling(window).mean(), color='green', linewidth=1)
    ax2.tick_params(axis='y', labelcolor='green')
    
    #~~~~~~ Plot cov + nov aligned ~~~~~~
    PLOT = "cov_val_10"
    ax3 = axs[0, 1]
    ax3.set_title("COV 10 / NOV 10 Alignment", fontsize=10, pad=10)
    ax3.set_ylabel(PLOT, color='gray')
    ax3.yaxis.grid(True, color='lightgray', linewidth=0.5)
    ax3.tick_params(axis='y', labelsize=8)
    ax3.axvline(x=points[dataset][model]['111'], color='gray', linewidth=1)
    ax3.axvline(x=points[dataset][model]['base'], color='gray', linestyle="--", linewidth=1)

    ax3.plot(df0['steps'], df0[PLOT].rolling(window).mean(), color='black', linestyle=":", linewidth=2)
    for frame in [df1, df2, df3, df4, df5]:
        ax3.plot(frame['steps'], frame[PLOT].rolling(window).mean(), color='lightgray', linewidth=4)
    ax3.plot(df6['steps'], df6[PLOT].rolling(window).mean(), color='lightgray', linewidth=4)

    PLOT = "nov_val_10"
    ax4 = ax3.twinx()
    ax4.set_ylabel(PLOT, color='green')
    ax4.tick_params(axis='y', labelsize=8)

    ax4.plot(df0['steps'], df0[PLOT].rolling(window).mean(), color='green', linestyle='--', linewidth=1)
    for frame in [df1, df2, df3, df4, df5]:
        ax4.plot(frame['steps'], frame[PLOT].rolling(window).mean(), color='green', linewidth=1)
    ax4.plot(df6['steps'], df6[PLOT].rolling(window).mean(), color='green', linewidth=1)
    ax4.tick_params(axis='y', labelcolor='green')
    
    #~~~~~~ Plot hr + ndcg ~~~~~~   
    PLOT = "hr_test_10"
    ax5 = axs[1, 0]
    ax5.set_title("HR 10 / NDCG 10", fontsize=10, pad=10)
    ax5.set_ylabel(PLOT, color='blue')
    ax5.yaxis.grid(True, color='lightgray', linewidth=0.5)
    ax5.tick_params(axis='y', labelsize=8)
    ax5.axvline(x=points[dataset][model]['111'], color='gray', linewidth=1)
    ax5.axvline(x=points[dataset][model]['base'], color='gray', linestyle="--", linewidth=1)
   
    ax5.plot(df0['steps'], df0[PLOT].rolling(window).mean(), color='blue', linestyle="--", linewidth=1)
    for frame in [df1, df2, df3, df4, df5]:
        ax5.plot(frame['steps'], frame[PLOT].rolling(window).mean(), color='lightblue', linewidth=1)
    ax5.plot(df6['steps'], df6[PLOT].rolling(window).mean(), color='blue', linewidth=1)
    
    PLOT = "ndcg_test_10"
    ax6 = ax5.twinx()
    ax6.set_ylabel(PLOT, color='red')
    ax6.tick_params(axis='y', labelsize=8)

    
    ax6.plot(df0['steps'], df0[PLOT].rolling(window).mean(), color='red', linestyle="--", linewidth=1)
    for frame in [df1, df2, df3, df4, df5]:
        ax6.plot(frame['steps'], frame[PLOT].rolling(window).mean(), color='salmon', linewidth=1)
    ax6.plot(df6['steps'], df6[PLOT].rolling(window).mean(), color='red', linewidth=1)
    
    # ~~~~~~ Plot repetitivness ~~~~~~
    
    PLOT = "rep_test_5"
    ax7 = axs[1, 1]
    ax7.set_ylabel(PLOT, color='black')
    ax7.set_title("REP 5", fontsize=10, pad=10)
    ax7.yaxis.set_label_position("right")
    ax7.yaxis.tick_right()
    ax7.yaxis.grid(True, color='lightgray', linewidth=0.5)
    ax7.tick_params(axis='y', labelsize=8)
    ax7.axvline(x=points[dataset][model]['111'], color='gray', linewidth=1)
    ax7.axvline(x=points[dataset][model]['base'], color='gray', linestyle="--", linewidth=1)
    
    ax7.plot(df0['steps'], df0[PLOT].rolling(window).mean(), color='black', linestyle="--", linewidth=1)
    for frame in [df1, df2, df3, df4, df5]:
        ax7.plot(frame['steps'], frame[PLOT].rolling(window).mean(), color='gray', linewidth=1)
    ax7.plot(df6['steps'], df6[PLOT].rolling(window).mean(), color='black', linewidth=1)
    
    
    dataline = "loss"
    dfl0 = pd.read_pickle(f"{filepath}/base_{dataline}")
    dfl1 = pd.read_pickle(f"{filepath}/rl_001_{replica}_{dataline}")
    dfl2 = pd.read_pickle(f"{filepath}/rl_010_{replica}_{dataline}")
    dfl3 = pd.read_pickle(f"{filepath}/rl_011_{replica}_{dataline}")
    dfl4 = pd.read_pickle(f"{filepath}/rl_110_{replica}_{dataline}")
    dfl5 = pd.read_pickle(f"{filepath}/rl_101_{replica}_{dataline}")
    dfl6 = pd.read_pickle(f"{filepath}/rl_111_{replica}_{dataline}") 
    
    PLOT = "loss"
    ax8 = axs[2, 0]
    ax8.set_ylabel(PLOT, color='black')
    ax8.set_title("Loss", fontsize=10, pad=10)
    ax8.yaxis.set_label_position("left")
    ax8.set_ylim(2, 10)
    ax8.yaxis.grid(True, color='lightgray', linewidth=0.5)
    ax8.tick_params(axis='y', labelsize=8)
    ax8.yaxis.tick_left()

    ax8.plot(dfl0['steps'], dfl0[PLOT].rolling(5).mean(), color='black', linestyle="--", linewidth=1)
    for frame in [dfl1, dfl2, dfl3, dfl4, dfl5]:
        ax8.plot(frame['steps'], frame[PLOT].rolling(5).mean(), color='gray', linewidth=1)
    ax8.plot(dfl6['steps'], dfl6[PLOT].rolling(5).mean(), color='black', linewidth=1)
    
    PLOT = "plain"
    ax9 = axs[2, 1]
    ax9.set_ylabel("base", color='gray')
    ax9.set_title("Loss components", fontsize=10, pad=10)
    ax9.set_ylim(0, 7)
    ax9.yaxis.grid(True, color='lightgray', linewidth=0.5)

    for frame in [dfl1, dfl2, dfl3, dfl4, dfl5]:
        ax9.plot(frame['steps'], frame[PLOT].rolling(5).mean(), color='lightgray', linewidth=1)
    ax9.plot(dfl6['steps'], dfl6[PLOT].rolling(5).mean(), color='black', linewidth=1)

    PLOT = "smorl"
    ax10 = ax9.twinx()
    ax10.set_ylabel(PLOT, color='green')
    ax10.set_ylim(0, 7)


    for frame in [dfl1, dfl2, dfl3, dfl4, dfl5]:
        ax10.plot(frame['steps'], frame[PLOT].rolling(5).mean(), color='lightgreen', linewidth=1)
    ax10.plot(dfl6['steps'], dfl6[PLOT].rolling(5).mean(), color='green', linewidth=1)
    ax10.tick_params(axis='y', labelcolor='green')
    
    

basepath = "/home/marek/Kinit/my_smorl/Plots/Paparella_both/"

window = 1
dataset = "rc15_results"
model = "caser"
replica = "target"
dataline = "metrics"

plot_metrics(basepath, dataset, model, "target")

#for dataset in datasets:
#    for model in models: 
#        for replica in ['main', 'target']:
#            filepath = f"{basepath}/{dataset}/{model}"
            